# 05 - MedGemma baseline sur le final

Objectif : appliquer une seule fois le baseline historique `google/medgemma-4b-it` au meme split final que MedSigLIP. Le modele, le prompt, la generation et les garde-fous sont figes. Aucun resultat de ce notebook ne doit servir a modifier la configuration.

Ce notebook peut etre execute dans la session MedSigLIP existante apres liberation de sa memoire GPU, ou dans une nouvelle session disposant du meme fichier de selection.

In [ ]:
%pip install -q -U "transformers>=4.56" accelerate pillow pandas scikit-learn


In [ ]:
from pathlib import Path, PurePosixPath
import gc
import hashlib
import json
import os
import subprocess
import sys

import pandas as pd
import torch
from IPython.display import display

REPO_DIR = Path("/kaggle/working/ARVI-RX-DS-4B")
DATASET_ROOT = Path("/kaggle/input/datasets/ashery/chexpert")
SELECTION_CSV = Path("/kaggle/working/arvi_chexpert_selection.csv")
OUTPUT_DIR = Path("/kaggle/working/medgemma_baseline_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/alalab12/ARVI-RX-DS-4B.git", str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)

if not os.environ.get("HF_TOKEN"):
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")


In [ ]:
config_path = REPO_DIR / "config" / "medgemma_baseline_v1.json"
baseline_config = json.loads(config_path.read_text(encoding="utf-8"))
prompt_path = REPO_DIR / "prompts" / "baseline_prompt.txt"
prompt_bytes = prompt_path.read_bytes().replace(b"\r\n", b"\n").replace(b"\r", b"\n")
prompt_hash = hashlib.sha256(prompt_bytes).hexdigest()

assert baseline_config["version"] == "medgemma_baseline_v1"
assert baseline_config["model_id"] == "google/medgemma-4b-it"
assert prompt_hash == baseline_config["prompt_sha256_canonical_lf"]
assert baseline_config["generation"] == {"do_sample": False}
print(baseline_config)


In [ ]:
if "selection" not in globals():
    if not SELECTION_CSV.exists():
        raise FileNotFoundError(
            "Selection absente. Executer dans la session precedente ou fournir "
            f"le fichier {SELECTION_CSV}."
        )
    selection = pd.read_csv(SELECTION_CSV)

if "expected_label" not in selection and "project_label" in selection:
    selection = selection.rename(columns={"project_label": "expected_label"})

def resolve_image_path(raw_path):
    direct = Path(str(raw_path))
    if direct.exists():
        return direct
    parts = PurePosixPath(str(raw_path).replace("\\", "/")).parts
    if parts and parts[0] == "CheXpert-v1.0-small":
        parts = parts[1:]
    return DATASET_ROOT.joinpath(*parts)

final = selection.loc[selection["split"].eq("final")].copy()
dev = selection.loc[selection["split"].eq("dev")].copy()
assert len(final) == 30
if "patient_id" in selection:
    assert set(final["patient_id"]).isdisjoint(set(dev["patient_id"]))
if "case_id" not in final:
    final["case_id"] = [f"baseline_final_{index:03d}" for index in range(len(final))]
final["image_path_resolved"] = final["image_path"].map(resolve_image_path)
assert final["image_path_resolved"].map(Path.exists).all()
display(final["expected_label"].value_counts())


## Charger le baseline fige

La cellule suivante libere MedSigLIP s'il est encore en memoire. Les artefacts CSV deja sauvegardes ne sont pas affectes.

In [ ]:
globals().pop("model", None)
globals().pop("processor", None)
gc.collect()
torch.cuda.empty_cache()

os.environ["MODEL_BACKEND"] = "medgemma"
os.environ["MEDGEMMA_MODEL_ID"] = baseline_config["model_id"]
os.environ["MEDGEMMA_MAX_NEW_TOKENS"] = str(baseline_config["max_new_tokens"])
sys.path.insert(0, str(REPO_DIR))

from src.inference import get_medgemma_backend, predict

get_medgemma_backend.cache_clear()


## Evaluation finale

Passer `RUN_BASELINE_FINAL` a `True` une seule fois. Une erreur technique reste une erreur et n'est pas remplacee par une nouvelle configuration.

In [ ]:
RUN_BASELINE_FINAL = False

if RUN_BASELINE_FINAL:
    records = []
    for position, (_, row) in enumerate(final.iterrows(), start=1):
        try:
            prediction = predict(
                row["image_path_resolved"],
                mode=baseline_config["mode"],
                backend="medgemma",
            )
            record = row.to_dict()
            record.update({
                "predicted_class": prediction["predicted_class"],
                "raw_predicted_class": prediction.get("raw_predicted_class"),
                "confidence": prediction.get("confidence"),
                "raw_confidence": prediction.get("raw_confidence"),
                "image_quality": prediction.get("image_quality"),
                "guardrail_actions": json.dumps(prediction.get("guardrail_actions", [])),
                "latency_ms": prediction.get("latency_ms"),
                "prediction_json": json.dumps(prediction),
                "technical_error": "",
            })
        except Exception as error:
            record = row.to_dict()
            record.update({
                "predicted_class": "technical_error",
                "latency_ms": None,
                "prediction_json": "",
                "technical_error": str(error),
            })
        records.append(record)
        print(f"{position}/{len(final)}")

    baseline_final = pd.DataFrame(records)
    baseline_final["image_path_resolved"] = baseline_final["image_path_resolved"].astype(str)
    baseline_final.to_csv(OUTPUT_DIR / "medgemma_baseline_final_predictions.csv", index=False)
else:
    print("Baseline final verrouille. Passer RUN_BASELINE_FINAL a True une seule fois.")


In [ ]:
if RUN_BASELINE_FINAL:
    valid = baseline_final.loc[
        baseline_final["technical_error"].fillna("").eq("")
    ].copy()
    definitive = valid.loc[
        valid["expected_label"].isin(["normal", "suspected_opacity"])
    ].copy()
    positive = definitive["expected_label"].eq("suspected_opacity")
    negative = definitive["expected_label"].eq("normal")
    baseline_summary = pd.DataFrame([{
        "model_version": baseline_config["version"],
        "n_definitive": len(definitive),
        "strict_accuracy": definitive["predicted_class"].eq(definitive["expected_label"]).mean(),
        "opacity_sensitivity": definitive.loc[positive, "predicted_class"].eq("suspected_opacity").mean(),
        "opacity_to_normal_rate": definitive.loc[positive, "predicted_class"].eq("normal").mean(),
        "normal_specificity": definitive.loc[negative, "predicted_class"].eq("normal").mean(),
        "normal_to_opacity_rate": definitive.loc[negative, "predicted_class"].eq("suspected_opacity").mean(),
        "uncertain_rate": definitive["predicted_class"].eq("uncertain").mean(),
        "median_latency_ms": definitive["latency_ms"].median(),
        "technical_errors": len(baseline_final) - len(valid),
    }])
    baseline_summary.to_csv(OUTPUT_DIR / "medgemma_baseline_final_summary.csv", index=False)
    display(baseline_summary)
    display(pd.crosstab(valid["expected_label"], valid["predicted_class"]))
    print("Cohorte CheXpert -1, sans accuracy:")
    display(
        valid.loc[valid["expected_label"].eq("uncertain"), "predicted_class"]
        .value_counts(normalize=True, dropna=False)
    )
    baseline_errors = definitive.loc[
        definitive["predicted_class"].ne(definitive["expected_label"])
    ]
    display(baseline_errors[[
        "patient_id", "expected_label", "predicted_class",
        "confidence", "image_quality", "image_path"
    ]])
